In [19]:
import numpy as np
import geopandas as gpd
import rasterio as rio
from shapely.geometry import box
import math
from pathlib import Path
from tqdm import tqdm

In [18]:
names = locals()

In [7]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [ ]:
# Reproject to Eckert IV (equal-area) so that:
# (1) geometry.area returns m², (2) sjoin runs in a common CRS,
# (3) results align with downstream Eckert IV rasters.
quarries = gpd.read_file(DATA / 'global-quarries-210101.gpkg').to_crs('ESRI:54012')
maus = gpd.read_file(DATA / 'global_mining_polygons_v2.gpkg').to_crs('ESRI:54012')

In [ ]:
# Maus polygons represent non-aggregate mines (metals, coal, industrial minerals).
# Any OSM quarry intersecting them is excluded from the construction-aggregate set.
# Left join keeps every OSM quarry; index_right is non-null where overlap exists.
joined = gpd.sjoin(quarries, maus[['geometry']], how='left', predicate='intersects')

# A quarry intersecting multiple Maus polygons appears in multiple rows;
# .unique() collapses to one ID per quarry.
overlap_ids = joined[joined.index_right.notna()].index.unique()

aggregate_quarries = quarries.drop(index=overlap_ids).copy()

In [15]:
print(f"Original: {len(quarries):,}")
print(f"Overlap with Maus (excluded): {len(overlap_ids):,}")
print(f"Aggregate-only retained: {len(aggregate_quarries):,}")

Original: 148,214
Overlap with Maus (excluded): 11,548
Aggregate-only retained: 136,666


In [ ]:
# Area in m² is valid because the CRS is equal-area.
aggregate_quarries['area_m2'] = aggregate_quarries.geometry.area
print(f"Total aggregate quarry area: {aggregate_quarries['area_m2'].sum()/1e6:.1f} km²")

Total aggregate quarry area: 19141.7 km²


In [22]:
aggregate_quarries.to_file(
    DATA / 'global-aggregate-quarries-210101.gpkg',
    driver='GPKG',
)